# 💧 AquaSense AI — 01: Exploratory Data Analysis (EDA)
**Project:** Intelligent Water Quality Assessment and Potability Prediction Using Explainable Machine Learning  
**Author:** Humza | CodeVibe with Humza  

### Overview:
In this notebook, we perform a deep exploratory analysis on the **Water Potability Dataset** (3,276 samples across 9 physicochemical parameters and 1 binary potability label).
We explore data types, missing value patterns, class imbalance, distributions, and correlation structure.


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.loader import DataLoader
from src.utils.config import WHO_STANDARDS, COLORS
from src.utils.visualization import set_plot_style

set_plot_style()
print("Libraries imported successfully.")


## 1. Load Dataset & Schema Verification


In [ ]:
dl = DataLoader(data_path="../data/raw/water_potability.csv")
df = dl.load()
summary = dl.get_summary(df)

print(f"Dataset Shape: {df.shape}")
print(f"Total Rows: {summary['rows']}, Total Columns: {summary['columns']}")
df.head(5)


## 2. Statistical Summary & Descriptive Metrics


In [ ]:
df.describe().T


## 3. Missing Value Analysis
Identify attributes with missing observations and calculate missingness percentages.


In [ ]:
missing_df = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage (%)': (df.isnull().mean() * 100).round(2)
})
missing_df = missing_df[missing_df['Missing Values'] > 0]
print(missing_df)

plt.figure(figsize=(8, 4))
sns.barplot(x=missing_df['Percentage (%)'], y=missing_df.index, palette='Blues_r')
plt.title("Missing Values Percentage by Feature", fontsize=12, color=COLORS['accent'])
plt.xlabel("Missing Percentage (%)")
plt.tight_layout()
plt.show()


## 4. Target Class Distribution (Imbalance Analysis)
Potability is coded as: `0 = Not Potable`, `1 = Potable`.


In [ ]:
counts = df['Potability'].value_counts()
pcts = df['Potability'].value_counts(normalize=True) * 100

print(f"Class 0 (Not Potable): {counts[0]} ({pcts[0]:.2f}%)")
print(f"Class 1 (Potable): {counts[1]} ({pcts[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(counts, labels=['Not Potable (0)', 'Potable (1)'], autopct='%1.1f%%',
       colors=[COLORS['danger'], COLORS['safe']], startangle=140, explode=[0.05, 0])
plt.title("Target Potability Class Distribution", fontsize=12, color=COLORS['accent'])
plt.tight_layout()
plt.show()


## 5. Feature Distributions & Kernel Density Estimation (KDE)
Examine distribution shape across classes for each water quality indicator.


In [ ]:
features = [c for c in df.columns if c != 'Potability']
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    sns.kdeplot(data=df[df['Potability'] == 0][feat], ax=ax, label='Not Potable (0)', color=COLORS['danger'], fill=True, alpha=0.3)
    sns.kdeplot(data=df[df['Potability'] == 1][feat], ax=ax, label='Potable (1)', color=COLORS['accent'], fill=True, alpha=0.3)
    ax.set_title(feat, color=COLORS['accent'], fontsize=11)
    ax.legend()

plt.suptitle("KDE Distribution by Potability Target", fontsize=15, color=COLORS['text_primary'], y=1.02)
plt.tight_layout()
plt.show()


## 6. Correlation Heatmap
Evaluate pairwise Pearson correlations to identify linear multicollinearity.


In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, linewidths=0.5)
plt.title("Feature Correlation Matrix", fontsize=13, color=COLORS['accent'])
plt.tight_layout()
plt.show()


## 7. Outlier Detection (Box Plots)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    sns.boxplot(y=df[feat], ax=ax, color=COLORS['accent'])
    ax.set_title(feat, color=COLORS['accent'])

plt.suptitle("Physicochemical Outlier Profiling", fontsize=15, color=COLORS['text_primary'], y=1.02)
plt.tight_layout()
plt.show()
